# 4.- Usando la imagen "colores.jpg" cuenta los elementos distinguiendo por colores

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

## Cargar y mostrar la imagen original

In [ ]:
# Cargar imagen
img_bgr = cv2.imread('colores.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'Dimensiones de la imagen: {img_rgb.shape}')

## Definir rangos de color en HSV y función para extraer objetos

In [ ]:
# Convertir a HSV para facilitar la segmentación por color
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

# Rangos HSV para cada color
color_ranges = {
    'Red': [
        (np.array([0, 80, 80]),   np.array([10, 255, 255])),
        (np.array([160, 80, 80]), np.array([180, 255, 255]))
    ],
    'Orange': [
        (np.array([10, 120, 100]), np.array([22, 255, 255]))
    ],
    'Yellow': [
        (np.array([22, 80, 150]),  np.array([35, 255, 255]))
    ],
    'Green': [
        (np.array([35, 60, 60]),   np.array([90, 255, 255]))
    ],
    'Blue': [
        (np.array([90, 60, 60]),   np.array([140, 255, 255]))
    ]
}

# Colores para visualización en matplotlib (R, G, B)
display_colors = {
    'Red':    'red',
    'Orange': 'orange',
    'Yellow': 'gold',
    'Green':  'green',
    'Blue':   'blue'
}

def get_color_mask(hsv_image, ranges):
    """Obtiene la máscara combinada para un color que puede tener múltiples rangos."""
    mask = np.zeros(hsv_image.shape[:2], dtype=np.uint8)
    for (lower, upper) in ranges:
        mask |= cv2.inRange(hsv_image, lower, upper)
    return mask

def count_objects(mask, min_area=800):
    """
    Aplica operaciones morfológicas y detecta contornos.
    Devuelve (contornos_válidos, máscara_limpia).
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN,  kernel, iterations=1)

    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = [c for c in contours if cv2.contourArea(c) >= min_area]
    return valid, cleaned

print('Rangos de color y funciones definidas correctamente.')

## Contar objetos por color y visualizar resultados

In [ ]:
results = {}  # {color_name: (count, contours, mask)}

for color_name, ranges in color_ranges.items():
    mask = get_color_mask(img_hsv, ranges)
    contours, cleaned_mask = count_objects(mask)
    results[color_name] = (len(contours), contours, cleaned_mask)
    print(f'{color_name} objects: {len(contours)}')

print('\nConteo completado.')

## Visualización: imagen original + objetos segmentados por color

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# --- Panel 0: Imagen original ---
axes[0].imshow(img_rgb)
axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

# --- Paneles 1-5: un color por panel ---
for idx, (color_name, (count, contours, mask)) in enumerate(results.items(), start=1):
    # Crear imagen segmentada: píxeles del color original, fondo blanco
    color_img = np.ones_like(img_rgb) * 255          # fondo blanco
    color_img[mask > 0] = img_rgb[mask > 0]          # conservar píxeles del color

    axes[idx].imshow(color_img)
    axes[idx].set_title(
        f'{color_name} objects: {count}',
        fontsize=14, fontweight='bold',
        color=display_colors[color_name]
    )
    axes[idx].axis('off')

plt.suptitle('Conteo de elementos por color', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('resultado_colores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada como resultado_colores.png')

## Visualización: imagen original con contornos marcados por color

In [ ]:
# Colores BGR para cv2.drawContours
bgr_colors = {
    'Red':    (0,   0,   220),
    'Orange': (0,   140, 255),
    'Yellow': (0,   220, 220),
    'Green':  (0,   180, 0),
    'Blue':   (220, 80,  0)
}

annotated = img_bgr.copy()
for color_name, (count, contours, _) in results.items():
    cv2.drawContours(annotated, contours, -1, bgr_colors[color_name], 3)

annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 10))
plt.imshow(annotated_rgb)
plt.title('Objetos detectados por color (contornos)', fontsize=15, fontweight='bold')
plt.axis('off')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=display_colors[c], label=f'{c}: {results[c][0]} obj.')
    for c in results
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=12,
           framealpha=0.85, title='Colores detectados')

plt.tight_layout()
plt.savefig('contornos_colores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada como contornos_colores.png')

## Resumen final del conteo

In [ ]:
print('='*40)
print('     RESUMEN - Objetos por color')
print('='*40)
total = 0
for color_name, (count, _, __) in results.items():
    print(f'  {color_name:<10}: {count:>3} objeto(s)')
    total += count
print('-'*40)
print(f'  TOTAL      : {total:>3} objeto(s)')
print('='*40)